# 06. cal01_base_v1 만들기

| 항목 | 내용 |
| --- | --- |
| 입력 | 원본 `T-CR1-CAL01_온도.csv`, `G-02_조업이벤트.csv` |
| 출력 | `data/processed/cal01_base_v1.csv` (원기록 345,390행) |
| 이 버전에서 하는 일 | 강종, 제외 표시, 구역별 `STAT_LABEL`을 붙임 |

**라벨 규칙**

| 컬럼 | 1 | 0 | 빈칸 |
| --- | --- | --- | --- |
| `구역__STAT_LABEL` | 구역 센서 중 하나라도 \|Z\| > 3 | 구역 센서가 모두 \|Z\| ≤ 3 | 제외 행, 센서값이 빈 행 |
| `구역__DOMAIN_LABEL` | v1에는 없음 | v1에는 없음 | 전부 빈칸 |

- Z는 같은 LOT 안에서, 제외 행을 뺀 평균과 표준편차로 계산함
- `LABEL`은 CSV에 저장하지 않음. 모델 노트북에서 두 라벨로 만듦

**v1 ~ v5 한눈에 보기** (버전마다 한 가지씩만 더함)

| 버전 | 단위 | STAT_LABEL | DOMAIN_LABEL | 만드는 노트북 |
| --- | --- | --- | --- | --- |
| v1 | 원기록 (1초) | 같은 LOT 기준 \|Z\| > 3 | 없음 | 06 |
| v2 | 원기록 (1초) | v1과 같음 | 공통 움직임 검토로 해제한 곳 0 | 07 |
| v3 | 원기록 (1초) | v1과 같음 | v2 + LOT 종료부 검토로 해제한 곳 0 | 08 |
| v4 | 1분 | 1분 안에 후보가 하나라도 있으면 1 | v2를 1분으로 묶음 | 09 |
| v5 | 1분 | v4와 같음 | v3를 1분으로 묶음 | 10 |

## 1. 공통 경로 (06은 원본만 읽음)

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v5)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v1.csv"  # 06은 가공 CSV를 읽지 않음
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 만들 CSV
OUTPUT_FILE = "cal01_base_v1.csv"
OUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)
os.makedirs(PROCESSED_DIR, exist_ok=True)

import numpy as np

## 2. 원본 읽기

In [ ]:
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)

temp["MEAS_DT"] = pd.to_datetime(temp["MEAS_DT"], errors="raise")
event["EVT_DT"] = pd.to_datetime(event["EVT_DT"], errors="raise")

# LOT 안에서 시간 순서로 정렬
temp = temp.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

print("원기록:", len(temp), "행")
print("LOT:", temp["LOT_NO"].nunique(), "개")

## 3. LOT에 강종 붙이기

- LOT의 첫 관측 시각과 같은 시각의 CR1 강종변경 기록을 찾음
- 그 기록의 바뀐 뒤 강종(`AFT_VAL`)을 LOT의 강종으로 씀

In [ ]:
# LOT별 첫 관측 시각
lot_start = temp.groupby("LOT_NO")["MEAS_DT"].min().reset_index()
lot_start.columns = ["LOT_NO", "START"]

# CR1 강종변경 기록만 고름
grade_change = event[(event["PLANT_CD"] == "CR1") & (event["EVT_TYPE"] == "강종변경")]
grade_change = grade_change[["EVT_DT", "AFT_VAL"]]
grade_change.columns = ["START", "GRADE"]

# 시각이 같은 기록끼리 연결
lot_grade = lot_start.merge(grade_change, on="START", how="left")

temp = temp.merge(lot_grade[["LOT_NO", "GRADE"]], on="LOT_NO", how="left")
lot_grade

## 4. 제외 구간 표시

팀이 정한 정기수리 전후 4개 구간에 `EXCLUDED=True`를 붙임. 행은 지우지 않음.

| LOT | 기준 시각 | 방향 | 이유 |
| --- | --- | --- | --- |
| 240058 | 06-09 08:00 | 이후 | 노 내림 |
| 240061 | 06-15 02:30 | 이전 | 승온 |
| 240114 | 11-04 20:00 | 이후 | 정기수리 전 하강 |
| 240117 | 11-13 09:00 | 이전 | 정기수리 후 승온 |

In [ ]:
USER_EXCLUSIONS = [
    (240058, "2024-06-09 08:00:00", "이후", "사용자 지정: 노 내림 구간 제외"),
    (240061, "2024-06-15 02:30:00", "이전", "사용자 지정: 승온 구간 제외"),
    (240114, "2024-11-04 20:00:00", "이후", "사용자 지정: 정기수리 전 하강 구간 제외"),
    (240117, "2024-11-13 09:00:00", "이전", "사용자 지정: 정기수리 후 승온 구간 제외"),
]

temp["EXCLUDED"] = False
temp["EXCLUSION_REASON"] = ""

for lot, boundary, direction, reason in USER_EXCLUSIONS:
    boundary = pd.Timestamp(boundary)

    if direction == "이후":
        in_time = temp["MEAS_DT"] >= boundary
    else:
        in_time = temp["MEAS_DT"] < boundary

    hit = (temp["LOT_NO"] == lot) & in_time

    temp.loc[hit, "EXCLUDED"] = True
    temp.loc[hit, "EXCLUSION_REASON"] = reason

    print(lot, reason, int(hit.sum()), "행")

## 5. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

## 6. 센서별 Z-score

- 같은 LOT 안에서 평균과 표준편차를 구함 (제외 행은 뺌)
- 표준편차는 ddof=0 (모든 행으로 나눔)

In [ ]:
base = temp

# 제외 행을 뺀 데이터로 LOT별 평균, 표준편차를 구함
use = base[base["EXCLUDED"] == False]
lot_mean = use.groupby("LOT_NO")[SENSORS].mean()
lot_std = use.groupby("LOT_NO")[SENSORS].std(ddof=0)

# 센서마다 |Z| = |값 - LOT 평균| / LOT 표준편차
z = pd.DataFrame(np.nan, index=base.index, columns=SENSORS)

for sensor in SENSORS:
    mean = base["LOT_NO"].map(lot_mean[sensor])
    std = base["LOT_NO"].map(lot_std[sensor])
    std = std.replace(0, np.nan)

    z[sensor] = (base[sensor] - mean).abs() / std

# 제외 행은 Z를 비워둠
z.loc[base["EXCLUDED"] == True] = np.nan

## 7. 구역별 라벨 붙이기

| 상황 | STAT_LABEL |
| --- | --- |
| 구역 센서 중 하나라도 \|Z\| > 3 | 1 |
| 구역 센서가 모두 \|Z\| ≤ 3 | 0 |
| 제외 행이거나 구역 센서값이 빔 | 빈칸 |

`DOMAIN_LABEL`은 v1에서 모두 빈칸.

In [ ]:
for zone in GROUPS:
    zone_sensors = GROUPS[zone]
    stat_col = zone + "__STAT_LABEL"
    domain_col = zone + "__DOMAIN_LABEL"

    # 구역 센서의 Z가 모두 있어야 판단할 수 있음
    available = z[zone_sensors].notna().all(axis=1)
    over = (z[zone_sensors] > 3).any(axis=1)

    base[stat_col] = np.nan
    base.loc[available, stat_col] = 0
    base.loc[available & over, stat_col] = 1

    base[domain_col] = np.nan

## 8. 저장

| 컬럼 묶음 | 컬럼 |
| --- | --- |
| 식별 | `ROW_ID`, `LOT_NO`, `MEAS_DT`, `GRADE` |
| 제외 | `EXCLUDED`, `EXCLUSION_REASON` |
| 센서 22개 | `CLN-IN`, `TC-Z1` ... `TC-OUT2`, `CP-Z4M` (CP-Z4M은 라벨 계산에 안 씀) |
| 라벨 22개 | `구역__STAT_LABEL`, `구역__DOMAIN_LABEL` |

In [ ]:
# 행 이름: LOT_시각 (예: 240001_20240102T033600)
base["ROW_ID"] = base["LOT_NO"].astype(str) + "_" + base["MEAS_DT"].dt.strftime("%Y%m%dT%H%M%S")

# 라벨을 1.0 대신 1로 저장하려고 정수형(Int64)으로 바꿈. Int64는 빈칸도 담을 수 있음
for column in LABEL_COLUMNS:
    base[column] = base[column].astype("Int64")

columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON"] + SAVE_SENSORS + LABEL_COLUMNS
base = base[columns]

base.to_csv(OUT_CSV, index=False, encoding=ENC)
print("저장:", OUTPUT_FILE, base.shape)

## 9. 구역별 라벨 개수

In [ ]:
# 구역별 개수
rows = []

for zone in GROUPS:
    stat = base[zone + "__STAT_LABEL"]
    domain = base[zone + "__DOMAIN_LABEL"]

    rows.append({
        "구역": zone,
        "STAT 1": int((stat == 1).sum()),
        "STAT 0": int((stat == 0).sum()),
        "STAT 빈칸": int(stat.isna().sum()),
        "DOMAIN 1": int((domain == 1).sum()),
        "DOMAIN 0": int((domain == 0).sum()),
    })

zone_counts = pd.DataFrame(rows)
zone_counts

## 10. LOT별 통계 후보 개수

In [ ]:
# LOT별로 구역마다 STAT_LABEL이 1인 개수
stat_columns = []
for zone in GROUPS:
    stat_columns.append(zone + "__STAT_LABEL")

lot_counts = base.groupby("LOT_NO")[stat_columns].sum()
lot_counts.columns = list(GROUPS)
lot_counts